# Week 9 — Pretrained Models: BERT, GPT, T5

Three architectures, three objectives, three philosophies. We dissect each, implement the pretraining objectives from scratch, and probe pretrained checkpoints to see what they have actually learned.

## Learning Objectives

- Explain encoder-only, decoder-only, and encoder–decoder architectures and which tasks each suits.
- Implement masked language modeling (MLM) and causal language modeling end-to-end.
- Implement T5-style span corruption.
- Probe a pretrained BERT layer-by-layer and reproduce the classic finding that linguistic abstractions emerge in order.

## Required Reading

- Devlin, J., et al. (2019). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*.
- Radford, A., et al. (2019). *Language Models are Unsupervised Multitask Learners* (GPT-2).
- Raffel, C., et al. (2020). *Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer* (T5).
- Rogers, A., Kovaleva, O., & Rumshisky, A. (2020). *A Primer in BERTology*.

In [ ]:
import sys, math, random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

torch.manual_seed(0); np.random.seed(0); random.seed(0)

## 1. Three architectures at a glance

| Model | Architecture | Pretraining objective | Best for |
|-------|--------------|----------------------|----------|
| **BERT** | Encoder only | Masked LM + NSP | Classification, NER, QA (extractive) |
| **GPT** | Decoder only (causal) | Next-token prediction | Generation, in-context learning |
| **T5** | Encoder–decoder | Span corruption | Any seq2seq task (translation, summarization, QA) |

The architectures differ only in their attention masks:

- BERT: **bidirectional** — every token attends to every other.
- GPT: **causal** — token $t$ attends only to $\leq t$.
- T5: **bidirectional encoder + causal decoder + cross-attention**.

## 2. Masked Language Modeling (BERT)

Given input $\mathbf{x}_{1:T}$, sample a random subset $M \subset \{1, \ldots, T\}$ with $|M|/T \approx 0.15$. For each $i \in M$:

- with probability 0.8, replace $x_i$ with `[MASK]`;
- with probability 0.1, replace $x_i$ with a random token;
- with probability 0.1, leave $x_i$ unchanged.

The 80/10/10 mixture reduces train–inference mismatch (no `[MASK]` token exists at inference). The loss is cross-entropy on the *masked positions only*.

$$\mathcal{L}_{\text{MLM}} = -\sum_{i \in M} \log P(x_i \mid \mathbf{x}_{\setminus M}).$$

In [ ]:
# Reuse the Transformer encoder we built in Week 8 (with a few simplifications inline).
def scaled_dot(Q, K, V, mask=None):
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(Q.size(-1))
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    return F.softmax(scores, dim=-1) @ V

class MHA(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.dk = h, d // h
        self.qkv = nn.Linear(d, 3 * d); self.o = nn.Linear(d, d)
    def forward(self, x, mask=None):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.h, self.dk).transpose(1, 2)
        k = k.view(B, T, self.h, self.dk).transpose(1, 2)
        v = v.view(B, T, self.h, self.dk).transpose(1, 2)
        out = scaled_dot(q, k, v, mask).transpose(1, 2).contiguous().view(B, T, D)
        return self.o(out)

class Block(nn.Module):
    def __init__(self, d, h, ff, causal=False):
        super().__init__()
        self.ln1 = nn.LayerNorm(d); self.attn = MHA(d, h)
        self.ln2 = nn.LayerNorm(d); self.ffn = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Linear(ff, d))
        self.causal = causal
    def forward(self, x, pad_mask=None):
        T = x.size(1)
        mask = None
        if pad_mask is not None:
            mask = pad_mask.unsqueeze(1).unsqueeze(2)
        if self.causal:
            causal = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), 1)
            mask = causal if mask is None else (mask | causal)
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.ffn(self.ln2(x))
        return x

class BERT(nn.Module):
    def __init__(self, V, d=64, h=4, ff=256, n_layers=3, max_len=64, pad=0):
        super().__init__()
        self.pad = pad
        self.emb = nn.Embedding(V, d, padding_idx=pad)
        self.pos = nn.Embedding(max_len, d)
        self.blocks = nn.ModuleList([Block(d, h, ff, causal=False) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(d)
        self.head = nn.Linear(d, V)
    def forward(self, x, return_hidden=False):
        T = x.size(1)
        pad_mask = (x == self.pad)
        h = self.emb(x) + self.pos(torch.arange(T, device=x.device))
        hiddens = [h]
        for block in self.blocks:
            h = block(h, pad_mask); hiddens.append(h)
        h = self.ln(h)
        logits = self.head(h)
        return (logits, hiddens) if return_hidden else logits

In [ ]:
# A toy "language" with a small grammar — enough to see MLM in action.
SENTENCES = []
subjects = ['cat', 'dog', 'bird', 'fox', 'student', 'teacher']
verbs    = ['saw', 'chased', 'liked', 'feared', 'ignored', 'followed']
objects  = ['ball', 'book', 'tree', 'house', 'mouse', 'idea']
adv      = ['quickly', 'silently', 'happily', 'reluctantly']

for s in subjects:
    for v in verbs:
        for o in objects:
            SENTENCES.append(f"the {s} {v} the {o}")
            SENTENCES.append(f"the {s} {np.random.choice(adv)} {v} the {o}")

random.shuffle(SENTENCES)
SENTENCES = SENTENCES[:600]

words = sorted(set(w for s in SENTENCES for w in s.split()))
SPECIALS = ['<pad>', '[MASK]', '[CLS]', '[SEP]']
VOCAB = SPECIALS + words
tok2id = {t: i for i, t in enumerate(VOCAB)}
id2tok = {i: t for t, i in tok2id.items()}
V = len(VOCAB)
PAD, MASK, CLS, SEP = tok2id['<pad>'], tok2id['[MASK]'], tok2id['[CLS]'], tok2id['[SEP]']
print(f"|V| = {V}, |corpus| = {len(SENTENCES)}")

def encode(sent, max_len=12):
    ids = [CLS] + [tok2id[w] for w in sent.split()] + [SEP]
    ids = ids[:max_len] + [PAD] * max(0, max_len - len(ids))
    return ids

In [ ]:
def make_mlm_batch(sentences, batch_size, mask_rate=0.15, max_len=12):
    sents = random.sample(sentences, batch_size)
    inputs = torch.tensor([encode(s, max_len) for s in sents])
    labels = inputs.clone()
    rand = torch.rand(inputs.shape)
    # Only consider non-special tokens for masking.
    eligible = (inputs >= len(SPECIALS))
    mask = (rand < mask_rate) & eligible
    # 80/10/10
    r2 = torch.rand(inputs.shape)
    inputs_masked = inputs.clone()
    inputs_masked[mask & (r2 < 0.8)] = MASK
    rand_pos = mask & (r2 >= 0.8) & (r2 < 0.9)
    inputs_masked[rand_pos] = torch.randint(len(SPECIALS), V, inputs.shape)[rand_pos]
    # Loss computed only at masked positions
    labels[~mask] = -100
    return inputs_masked, labels

bert = BERT(V, d=64, h=4, ff=256, n_layers=3, max_len=12, pad=PAD)
opt = torch.optim.Adam(bert.parameters(), lr=3e-3)

losses = []
for step in range(400):
    x, y = make_mlm_batch(SENTENCES, batch_size=32)
    logits = bert(x)
    loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1), ignore_index=-100)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
    if (step + 1) % 100 == 0:
        print(f"step {step+1:4d}  MLM loss = {loss.item():.3f}")

plt.figure(figsize=(8, 3))
plt.plot(losses); plt.xlabel('step'); plt.ylabel('MLM loss')
plt.title('MLM training loss'); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Inspect: what does the model predict at a masked position?
bert.eval()
sent = "the cat [MASK] the mouse"
ids = [CLS] + [tok2id.get(w, MASK) for w in sent.split()] + [SEP]
ids = ids + [PAD] * (12 - len(ids))
x = torch.tensor([ids])
with torch.no_grad():
    logits = bert(x)
mask_pos = ids.index(MASK)
probs = F.softmax(logits[0, mask_pos], dim=-1)
top = torch.topk(probs, 5)
print(f"Masked position {mask_pos}: top-5 predictions for '{sent}':")
for p, i in zip(top.values, top.indices):
    print(f"  {id2tok[i.item()]:<12} p = {p.item():.3f}")

## 3. Causal Language Modeling (GPT)

Same Transformer block, **causal mask** on self-attention. Loss is next-token prediction over every position:

$$\mathcal{L}_{\text{CLM}} = -\sum_{t=1}^{T-1} \log P(x_{t+1} \mid x_{\leq t}).$$

No `[MASK]` token, no auxiliary objectives, no special bookkeeping. This simplicity is part of why GPT scaled so well.

In [ ]:
class GPT(nn.Module):
    def __init__(self, V, d=64, h=4, ff=256, n_layers=3, max_len=16, pad=0):
        super().__init__()
        self.pad = pad
        self.emb = nn.Embedding(V, d, padding_idx=pad)
        self.pos = nn.Embedding(max_len, d)
        self.blocks = nn.ModuleList([Block(d, h, ff, causal=True) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(d)
        self.head = nn.Linear(d, V)
    def forward(self, x):
        T = x.size(1)
        pad_mask = (x == self.pad)
        h = self.emb(x) + self.pos(torch.arange(T, device=x.device))
        for block in self.blocks:
            h = block(h, pad_mask)
        return self.head(self.ln(h))

def make_clm_batch(sentences, batch_size, max_len=14):
    sents = random.sample(sentences, batch_size)
    seqs = []
    for s in sents:
        ids = [CLS] + [tok2id[w] for w in s.split()] + [SEP]
        ids = ids[:max_len] + [PAD] * max(0, max_len - len(ids))
        seqs.append(ids)
    seqs = torch.tensor(seqs)
    return seqs[:, :-1], seqs[:, 1:]

gpt = GPT(V, d=64, h=4, ff=256, n_layers=3, max_len=14, pad=PAD)
opt = torch.optim.Adam(gpt.parameters(), lr=3e-3)
for step in range(400):
    x, y = make_clm_batch(SENTENCES, batch_size=32)
    logits = gpt(x)
    loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1), ignore_index=PAD)
    opt.zero_grad(); loss.backward(); opt.step()
    if (step + 1) % 100 == 0:
        print(f"step {step+1:4d}  CLM loss = {loss.item():.3f}")

In [ ]:
# Generation
gpt.eval()
prompt = "[CLS] the cat"
ids = [tok2id[w] for w in prompt.split() if w in tok2id]
for _ in range(8):
    x = torch.tensor([ids + [PAD] * (13 - len(ids))])
    with torch.no_grad():
        logits = gpt(x)
    next_id = torch.multinomial(F.softmax(logits[0, len(ids) - 1] / 0.7, dim=-1), 1).item()
    if next_id == SEP or next_id == PAD:
        break
    ids.append(next_id)
print("Generated:", ' '.join(id2tok[i] for i in ids if i not in (PAD,)))

## 4. T5-style span corruption

T5 (Raffel et al., 2020) uses a single text-to-text objective. During pretraining, contiguous spans of input tokens are replaced with sentinel tokens `<extra_id_0>`, `<extra_id_1>`, ..., and the model is trained to *generate the missing spans*, separated by the same sentinels.

> **Input:**  the cat `<X>` the `<Y>` mouse
> **Target:** `<X>` chased `<Y>` little `<Z>`

This is conceptually MLM but framed as seq2seq generation, which lets T5 handle any task uniformly.

In [ ]:
# Just the data preparation — running an encoder-decoder pretraining loop
# is structurally identical to Week 8 with this masking applied.
SENTINEL_BASE = V  # we extend the conceptual vocab
def make_span_corruption_example(sent, span_prob=0.15, mean_span=2):
    tokens = sent.split()
    n = len(tokens)
    keep = np.ones(n, dtype=bool)
    i = 0
    while i < n:
        if np.random.random() < span_prob:
            span_len = max(1, int(np.random.exponential(mean_span)))
            keep[i:i+span_len] = False
            i += span_len
        else:
            i += 1
    # Build source (with sentinels) and target (sentinel + dropped spans).
    src_tokens, tgt_tokens = [], []
    sentinel_idx = 0
    i = 0
    while i < n:
        if keep[i]:
            src_tokens.append(tokens[i]); i += 1
        else:
            src_tokens.append(f"<X{sentinel_idx}>")
            tgt_tokens.append(f"<X{sentinel_idx}>")
            while i < n and not keep[i]:
                tgt_tokens.append(tokens[i]); i += 1
            sentinel_idx += 1
    tgt_tokens.append(f"<X{sentinel_idx}>")
    return ' '.join(src_tokens), ' '.join(tgt_tokens)

for sent in random.sample(SENTENCES, 3):
    src, tgt = make_span_corruption_example(sent)
    print(f"orig  : {sent}")
    print(f"input : {src}")
    print(f"target: {tgt}\n")

## 5. Probing — what does BERT learn?

A linear probe: freeze BERT, train a small linear classifier on each layer's representations to predict some linguistic property (e.g., part of speech). If accuracy peaks at layer $\ell$, that's where the model represents the property most cleanly.

The classic finding (Tenney et al., 2019; Rogers et al., 2020): **BERT recovers the linguistic pipeline in order** — surface features at the bottom, syntax in the middle, semantics near the top.

We demonstrate with a tiny POS-style probe on our toy vocabulary.

In [ ]:
# Tag each word with a category: SUBJECT, VERB, OBJECT, ADV, DET, OTHER.
TAG = {}
for w in subjects: TAG[w] = 0   # SUBJECT
for w in verbs:    TAG[w] = 1   # VERB
for w in objects:  TAG[w] = 2   # OBJECT
for w in adv:      TAG[w] = 3   # ADV
TAG['the'] = 4                  # DET
for w in SPECIALS: TAG[w] = 5   # OTHER

bert.eval()
# Collect hidden states layer-by-layer + their gold tags.
hiddens_by_layer = {l: [] for l in range(len(bert.blocks) + 1)}
tags = []
with torch.no_grad():
    for sent in SENTENCES[:200]:
        ids = encode(sent)
        x = torch.tensor([ids])
        _, hs = bert(x, return_hidden=True)
        for l, h in enumerate(hs):
            for t in range(len(ids)):
                if ids[t] != PAD:
                    hiddens_by_layer[l].append(h[0, t].numpy())
        for t in range(len(ids)):
            if ids[t] != PAD:
                tags.append(TAG[id2tok[ids[t]]])

tags = np.array(tags)

# Simple linear probe: closed-form least-squares classifier.
from numpy.linalg import lstsq
def probe_accuracy(H, y, n_classes=6):
    Y = np.eye(n_classes)[y]
    H_bias = np.hstack([H, np.ones((len(H), 1))])
    W, *_ = lstsq(H_bias, Y, rcond=None)
    preds = (H_bias @ W).argmax(axis=1)
    return (preds == y).mean()

accs = []
for l, hs in hiddens_by_layer.items():
    H = np.stack(hs)
    accs.append(probe_accuracy(H, tags))
    print(f"layer {l}: probe acc = {accs[-1]:.3f}")

plt.figure(figsize=(7, 4))
plt.plot(list(hiddens_by_layer.keys()), accs, marker='o')
plt.xlabel('layer (0 = embedding)'); plt.ylabel('probe accuracy')
plt.title('Linear probe: word-class recovery vs. layer depth')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 6. Comparing the three objectives on a fixed-budget setting

A useful exercise (Liu et al., 2019; Raffel et al., 2020): hold parameters, training tokens, and architecture fixed, vary only the pretraining objective. The literature consensus:

- **MLM** is best for understanding tasks (GLUE, SuperGLUE).
- **CLM** is best for generation and shows the strongest scaling to model size.
- **Span corruption** matches or exceeds MLM, and unlike MLM the pretrained model is *directly* usable for any task without architecture changes.

Modern frontier models are predominantly CLM (decoder-only) for engineering reasons: simpler, scales to long contexts, KV caching at inference.

## 7. Exercises

1. **Pretrain a small BERT and measure GLUE-style transfer.** Pretrain 6 layers on 1M sentences, then fine-tune on a binary sentiment task with 1k, 10k, and 100k examples. Plot fine-tuned accuracy vs. pretraining steps.
2. **GPT vs. BERT on sentence similarity.** GPT was not designed for sentence embeddings, but with mean-pooling can still produce them. Compare both, then explain why BERT typically wins zero-shot but GPT can catch up with prompting (Reimers & Gurevych, 2019; Jiang et al., 2022).
3. **Span corruption ablation.** Vary span length (geometric mean $\in \{1, 3, 5, 10\}$) in T5-style corruption. Measure downstream task transfer. Where does mean span = 3 (T5's default) come from empirically?
4. **Probing layer order.** Reproduce Tenney et al. (2019): probe for POS, dependency, NER, coreference. Verify that the optimal layer for each task increases in roughly that order.

---

## Next Week

Week 10 — Fine-tuning and alignment. Full fine-tuning, LoRA, RLHF, and DPO.